Один раз подключаем GDrive, если ещё не сделано. Там уже должен находиться датасет (wav, 22050Hz, 16бит, моно) и прочие используемые данные.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get update
!apt-get install -y build-essential cmake ninja-build

In [ ]:
!git clone https://github.com/OHF-voice/piper1-gpl.git
%cd piper1-gpl

In [ ]:
!python3 -m venv .venv --without-pip
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!.venv/bin/python3 get-pip.py

In [ ]:
!.venv/bin/pip install --upgrade pip
!.venv/bin/pip install -e .[train]

In [ ]:
!bash build_monotonic_align.sh

In [ ]:
!.venv/bin/pip install scikit-build

In [ ]:
!.venv/bin/python setup.py build_ext --inplace

In [ ]:
!ls /content/piper1-gpl/src/piper/train/vits/monotonic_align

[Опциональное действие] Исправление фонем, предварительно скачать с репозитория https://github.com/mitrokun/espeak-ng-data папку espeak-ng-data на gdrive после чего запустить копирование данных. Если не планируете вмешиваться в piper в дальнейшем, то пропустите шаг.

In [10]:
!cp -r /content/drive/MyDrive/espeak-ng-data/* /content/piper1-gpl/src/piper/espeak-ng-data/

Остановка обучения производится вручную, следите за лимитом времени, при израсходывание лимита все данные удалятся. Если проводите дообучение, указывайте предварительно сохраненный чекпоинт с drive.
Перед выключением, сразу после создания чекпоинта, дайте завершить обучение одной эпохи (чекпоинт xx19, прерывайте на xx21, а не на xx20).

Пути на чекпоинты с HF "https://huggingface.co/datasets/rhasspy/piper-checkpoints/blob/main/ru/ru_RU/ruslan/medium/epoch%3D2436-step%3D1724372.ckpt" или "https://huggingface.co/datasets/rhasspy/piper-checkpoints/blob/main/ru/ru_RU/irina/medium/epoch%3D4139-step%3D929464.ckpt"

In [ ]:
!.venv/bin/python -m piper.train fit \
  --data.voice_name "ru_RU-sushko-medium" \
  --data.csv_path "/content/drive/MyDrive/yourvoice/metadata.csv" \
  --data.audio_dir "/content/drive/MyDrive/yourvoice/" \
  --model.sample_rate 22050 \
  --data.espeak_voice "ru" \
  --data.cache_dir "/content/cache/" \
  --data.config_path "/content/drive/MyDrive/ru_RU-yourvoice-medium.json" \
  --data.batch_size 8 \
  --ckpt_path "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/ruslan/medium/epoch%3D2436-step%3D1724372.ckpt" \
  --trainer.check_val_every_n_epoch 50 \
  --trainer.log_every_n_steps 1 \
  --data.num_workers 2

In [ ]:
!.venv/bin/python -m piper.train fit \
  --data.voice_name "ru_RU-glados-medium" \
  --data.csv_path "/content/drive/MyDrive/glados20/metadata.csv" \
  --data.audio_dir "/content/drive/MyDrive/glados20/" \
  --model.sample_rate 22050 \
  --data.espeak_voice "ru" \
  --data.cache_dir "/content/cache/" \
  --data.config_path "/content/drive/MyDrive/ru_RU-glados-medium.json" \
  --data.batch_size 8 \
  --ckpt_path "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/irina/medium/epoch%3D4139-step%3D929464.ckpt" \
  --trainer.check_val_every_n_epoch 50 \
  --trainer.log_every_n_steps 1 \
  --data.num_workers 2

После остановки обучения сохранить чекпоинт на drive, если требуется

In [ ]:
!cp /content/piper1-gpl/lightning_logs/version_0/checkpoints/epoch=*-step=*.ckpt /content/drive/MyDrive/pt/

Для экспорта  запустите действие, оно подхватит чекпоинт из каталога version_0

In [ ]:
!.venv/bin/python -m piper.train.export_onnx \
  --checkpoint /content/piper1-gpl/lightning_logs/version_0/checkpoints/epoch=*-step=*.ckpt \
  --output-file /content/drive/MyDrive/ru_RU-sushko-medium.onnx

Если не работает, то проблемы с обновленными библиотеками. Обычный день в мире python. Надо понизить торч, после чего повторите конвертацию. Если и это не помогло, идём в репозиторий piper1-gpl и ищем в issue актуальную информацию

In [ ]:
!.venv/bin/python -m pip uninstall torch -y

!.venv/bin/python -m pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!.venv/bin/python -m pip install onnxscript

In [ ]:
!mkdir cache
!cp -r /content/drive/MyDrive/cache/* /content/cache/

In [ ]:
!cp /content/cache/* /content/drive/MyDrive/cache

In [ ]:
!cp /content/piper1-gpl/lightning_logs/* /content/drive/MyDrive/


Если требуется отчистить кэш

In [ ]:
!rm -rf /content/cache